# Recommendation Engine EDA

Explore interaction density, item popularity, and sparsity patterns that drive recommender design.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
base_path = Path('../data/raw')
interactions = pd.read_csv(base_path / 'interactions.csv')
items = pd.read_csv(base_path / 'items.csv')
interactions.head()

In [ ]:
# Dataset overview
print(f'Interactions: {interactions.shape[0]} rows, {interactions.shape[1]} columns')
print(f'Items: {items.shape[0]} rows, {items.shape[1]} columns')
display(interactions.info())
display(items.info())
display(interactions.describe(include='all').T)

In [ ]:
# User activity distribution
activity = interactions.groupby('user_id').size().sort_values(ascending=False)
plt.figure(figsize=(10, 4))
sns.histplot(activity, bins=10, kde=True, color='steelblue')
plt.title('User activity distribution')
plt.xlabel('Interactions per user')
plt.tight_layout()
plt.show()

In [ ]:
# Popular items
popular = interactions.groupby('item_id').agg(interaction_count=('item_id', 'size'), avg_rating=('rating', 'mean')).sort_values(['interaction_count', 'avg_rating'], ascending=False).head(10)
popular = popular.join(items.set_index('item_id')[['title', 'genres']], how='left')
display(popular)
plt.figure(figsize=(10, 5))
sns.barplot(x=popular['interaction_count'], y=popular['title'], palette='viridis')
plt.title('Top popular items')
plt.xlabel('Interaction count')
plt.ylabel('Item')
plt.tight_layout()
plt.show()

In [ ]:
# Sparsity analysis
matrix = interactions.pivot_table(index='user_id', columns='item_id', values='rating', aggfunc='mean', fill_value=0)
sparsity = 1 - (matrix.astype(bool).sum().sum() / (matrix.shape[0] * matrix.shape[1]))
print(f'User-item matrix shape: {matrix.shape}')
print(f'Sparsity: {sparsity:.2%}')
plt.figure(figsize=(8, 5))
sns.heatmap(matrix.corr(), cmap='Blues', linewidths=0.3)
plt.title('Item correlation heatmap')
plt.tight_layout()
plt.show()

## Key Insights

- Sparse user-item matrices make fallback strategies necessary.
- Popular items should remain part of the hybrid scorer.
- Collaborative and content-based signals are both useful when interactions are limited.

In [ ]:
# Recommendation quality diagnostics: catalog coverage and long-tail concentration
item_popularity = interactions.groupby('item_id').size().sort_values(ascending=False)
coverage_ratio = item_popularity.size / items['item_id'].nunique()
head_share = item_popularity.head(max(1, int(0.2 * len(item_popularity)))).sum() / item_popularity.sum()

print(f'Catalog coverage in interactions: {coverage_ratio:.2%}')
print(f'Top-20% items share of interactions: {head_share:.2%}')

plt.figure(figsize=(10, 4))
item_popularity.reset_index(drop=True).plot(kind='line', color='darkorange')
plt.title('Item popularity curve (head vs long-tail)')
plt.xlabel('Items sorted by popularity rank')
plt.ylabel('Interaction count')
plt.tight_layout()
plt.show()

## Recommendation Quality Insights

- A high head-share indicates popularity bias risk, so hybrid blending should preserve some exploration via content and collaborative signals.
- Lower catalog coverage usually means sparse feedback; cold-start defaults and popularity priors become critical.
- These EDA checks justify tuning hybrid weights rather than hardcoding them.